# 2D Grating Coupler Simulation with Au Mirror

This notebook sets up a 2D grating coupler simulation using Tidy3D.
The structure is based on an SOI (Silicon-on-Insulator) platform with:
- Silicon (Si) waveguide layer with cylindrical holes
- Buried oxide (SiO2) layer
- Silicon substrate
- Gold (Au) bottom mirror for improved upward coupling efficiency

Materials are sourced from the Tidy3D material library.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tidy3d as td
import tidy3d.web as web

td.config.logging_level = "ERROR"

## Simulation Parameters

In [ ]:
# Central wavelength (um)
wavelength = 1.55
freq0 = td.C_0 / wavelength

# Frequency bandwidth
fwidth = freq0 / 10

# Grating parameters
period = 0.630          # Grating period (um)
duty_cycle = 0.5        # Duty cycle (fraction of period with holes)
n_periods = 20          # Number of grating periods

# Hole geometry
hole_radius = (duty_cycle * period) / 2  # Radius of cylindrical holes (um)
etch_depth = 0.070       # Hole etch depth (um) -- partial etch

# SOI layer thicknesses (um)
si_thickness = 0.220     # Silicon waveguide layer thickness
box_thickness = 2.0      # Buried oxide (BOX) layer thickness
sub_thickness = 0.5      # Silicon substrate thickness (truncated for simulation)
au_thickness  = 0.100    # Gold mirror thickness

# Coupling angle (degrees from normal, positive toward waveguide)
theta_in = 8.0

# Padding for simulation domain edges (um)
x_pad = 1.0              # Padding in x (propagation direction) on each side
z_pad_top = 1.5          # Padding above structure
z_pad_bot = 0.1          # Padding below Au mirror

# Corner mirror shift (um) -- if applicable
mirror_shift = 0.0

# Simulation run time
run_time = 1e-12  # seconds

print(f"Hole radius: {hole_radius:.4f} um")
print(f"Grating total length: {n_periods * period:.3f} um")

## Materials from Tidy3D Material Library

In [ ]:
# Silicon (crystalline) -- waveguide core and substrate
si_medium = td.material_library["cSi"]["Li1993_293K"]

# Silicon dioxide -- buried oxide (BOX) layer
sio2_medium = td.material_library["SiO2"]["Palik_Lossless"]

# Gold -- bottom mirror
au_medium = td.material_library["Au"]["Olmon2012evaporated"]

# Air / cladding above grating
air_medium = td.Medium(permittivity=1.0)

print("Materials loaded from Tidy3D material library:")
print(f"  Si  : {si_medium}")
print(f"  SiO2: {sio2_medium}")
print(f"  Au  : {au_medium}")

## Geometry: SOI Structure with Cylindrical Holes and Au Mirror

In [ ]:
# --- Coordinate reference ---
# z = 0 at the top surface of the Si waveguide layer.
# x = 0 at the start of the grating.

grating_length = n_periods * period

# z-coordinates for layer interfaces
z_si_top    =  0.0
z_si_bot    =  z_si_top - si_thickness      # bottom of Si layer
z_box_bot   =  z_si_bot - box_thickness     # bottom of BOX
z_au_bot    =  z_box_bot - au_thickness     # bottom of Au mirror
z_sub_bot   =  z_au_bot  - sub_thickness    # bottom of Si substrate (simulation boundary)

# Simulation x-domain
x_min = -x_pad
x_max =  grating_length + x_pad
sim_x = x_max - x_min

# Simulation z-domain
z_min = z_au_bot - z_pad_bot
z_max = z_si_top + z_pad_top
sim_z = z_max - z_min

# Simulation y-domain (2D: very thin cell, one period in y for periodicity or just small width)
sim_y = 0.5  # thin slab; use periodic BCs in y

print(f"Simulation domain: x=[{x_min:.2f}, {x_max:.2f}] um, "
      f"z=[{z_min:.3f}, {z_max:.2f}] um, y=[{-sim_y/2:.2f}, {sim_y/2:.2f}] um")

# --- Structures ---

# 1. Si waveguide layer (slab)
si_slab = td.Structure(
    geometry=td.Box(
        center=[(x_min + x_max) / 2, 0, (z_si_top + z_si_bot) / 2],
        size=[sim_x, sim_y + 1, si_thickness],
    ),
    medium=si_medium,
    name="si_slab",
)

# 2. Buried oxide (BOX) layer
box_layer = td.Structure(
    geometry=td.Box(
        center=[(x_min + x_max) / 2, 0, (z_si_bot + z_box_bot) / 2],
        size=[sim_x, sim_y + 1, box_thickness],
    ),
    medium=sio2_medium,
    name="box_layer",
)

# 3. Gold (Au) bottom mirror
au_mirror = td.Structure(
    geometry=td.Box(
        center=[(x_min + x_max) / 2, 0, (z_box_bot + z_au_bot) / 2],
        size=[sim_x, sim_y + 1, au_thickness],
    ),
    medium=au_medium,
    name="au_mirror",
)

# 4. Si substrate (below Au mirror)
si_substrate = td.Structure(
    geometry=td.Box(
        center=[(x_min + x_max) / 2, 0, (z_au_bot + z_sub_bot) / 2],
        size=[sim_x, sim_y + 1, sub_thickness],
    ),
    medium=si_medium,
    name="si_substrate",
)

# 5. Cylindrical holes etched into the Si waveguide layer
holes = []
for i in range(n_periods):
    x_center = (i + 0.5) * period  # center of hole in x
    hole = td.Structure(
        geometry=td.Cylinder(
            center=[x_center, 0, z_si_top - etch_depth / 2],
            radius=hole_radius,
            length=etch_depth,
            axis=2,  # cylinder axis along z
        ),
        medium=air_medium,
        name=f"hole_{i}",
    )
    holes.append(hole)

structures = [si_slab, box_layer, au_mirror, si_substrate] + holes
print(f"Total structures: {len(structures)} ({len(holes)} holes)")

## Source and Monitors

In [ ]:
# --- Gaussian pulse source ---
pulse = td.GaussianPulse(freq0=freq0, fwidth=fwidth)

# Mode source injecting TE mode into the Si waveguide from the left (negative x)
mode_source = td.ModeSource(
    center=[x_min + 0.3, 0, (z_si_top + z_si_bot) / 2],
    size=[0, sim_y + 1, si_thickness * 4],
    source_time=pulse,
    direction="+",
    mode_spec=td.ModeSpec(num_modes=1),
    name="mode_source",
)

# --- Monitors ---

# Upward monitor (above grating, in air cladding)
z_upward = z_si_top + z_pad_top * 0.8
monitor_up = td.FluxMonitor(
    center=[(x_min + x_max) / 2, 0, z_upward],
    size=[grating_length, sim_y + 1, 0],
    freqs=[freq0],
    name="monitor_up",
    normal_dir="+",
)

# Downward monitor (at the top of the buried oxide, below Si slab)
z_downward = z_si_bot - 0.05  # just below the Si layer
monitor_down = td.FluxMonitor(
    center=[(x_min + x_max) / 2, 0, z_downward],
    size=[grating_length, sim_y + 1, 0],
    freqs=[freq0],
    name="monitor_down",
    normal_dir="-",
)

# Backward monitor (measures reflected power back into waveguide)
x_backward = x_min + 0.5
monitor_back = td.ModeMonitor(
    center=[x_backward, 0, (z_si_top + z_si_bot) / 2],
    size=[0, sim_y + 1, si_thickness * 4],
    freqs=[freq0],
    mode_spec=td.ModeSpec(num_modes=1),
    name="monitor_back",
)

# Field monitor for visualization (x-z plane)
field_monitor_xz = td.FieldMonitor(
    center=[(x_min + x_max) / 2, 0, (z_min + z_max) / 2],
    size=[sim_x, 0, sim_z],
    freqs=[freq0],
    name="field_xz",
)

# Field monitor for visualization (x-y plane at Si layer mid)
field_monitor_xy = td.FieldMonitor(
    center=[(x_min + x_max) / 2, 0, z_si_top - etch_depth / 2],
    size=[sim_x, sim_y, 0],
    freqs=[freq0],
    name="field_xy",
)

monitors = [monitor_up, monitor_down, monitor_back, field_monitor_xz, field_monitor_xy]
print("Monitors defined: upward, downward, backward, field_xz, field_xy")

## Create Simulation

In [ ]:
# Grid specification -- cells per wavelength in each medium
grid_spec = td.GridSpec.auto(
    min_steps_per_wvl=20,
    wavelength=wavelength,
)

# Boundary conditions:
# - Periodic in y (2D approximation)
# - PML in x and z
boundary_spec = td.BoundarySpec(
    x=td.Boundary.pml(),
    y=td.Boundary.periodic(),
    z=td.Boundary.pml(),
)

sim = td.Simulation(
    center=[(x_min + x_max) / 2, 0, (z_min + z_max) / 2],
    size=[sim_x, sim_y, sim_z],
    grid_spec=grid_spec,
    structures=structures,
    sources=[mode_source],
    monitors=monitors,
    run_time=run_time,
    boundary_spec=boundary_spec,
    medium=air_medium,
)

print("Simulation created successfully.")
print(f"  Domain size: {sim.size}")
print(f"  Grid cells : {sim.num_cells:,}")
print(f"  Run time   : {sim.run_time:.2e} s")

## Visualize Structure (x-z and x-y Slices)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# x-z cross-section at y=0
sim.plot(y=0, ax=axes[0])
axes[0].set_title("Structure: x-z cross-section (y=0)")
axes[0].set_xlabel("x (um)")
axes[0].set_ylabel("z (um)")

# x-y cross-section at mid Si layer (z = z_si_top - etch_depth/2)
z_mid_si = z_si_top - etch_depth / 2
sim.plot(z=z_mid_si, ax=axes[1])
axes[1].set_title(f"Structure: x-y cross-section (z={z_mid_si:.3f} um, mid-etch)")
axes[1].set_xlabel("x (um)")
axes[1].set_ylabel("y (um)")

plt.tight_layout()
plt.show()

## Estimate Simulation Credits

Before submitting the simulation to the cloud, estimate the number of FlexCredits it will consume.
This uploads the simulation job and queries the cost without actually running it.

In [ ]:
# Create a Job object (does not run the simulation)
job = web.Job(simulation=sim, task_name="2d_grating_coupler_au_mirror")

# Estimate the FlexCredit cost
# Note: cost is calculated assuming the simulation runs for the full run_time.
# If early shutoff is triggered, the actual cost will be lower.
estimated_cost = job.estimate_cost(verbose=True)

print(f"\nEstimated maximum cost: {estimated_cost:.4f} FlexCredits")